# 00 · Environment and smoke test

**Goal:** prove the full loop works before any real game is processed: Colab GPU, Google Drive, this repo, a video, a detector, and metrics written back.

**Runtime:** L4 GPU is enough for this notebook (about 1.7 compute units per hour). Expected run time: under 15 minutes.

**What it does**
1. Checks the GPU and records its name and memory.
2. Mounts Google Drive and creates the project folders (safe to re-run; nothing is overwritten).
3. Clones the repo and runs its tests.
4. Downloads the first 120 s of the public PlayOn sample game (high school, 1080p, used only to test the pipeline) into Drive.
5. Runs a pretrained RF-DETR person detector on sampled frames and measures speed.
6. Writes `results/00_smoke/metrics.json` to the repo and a copy to Drive.

**Pass criteria:** tests pass, video reads, detector finds 8 or more people in most frames, metrics file written.

In [ ]:
# 1. GPU check
import subprocess, json, time, os, platform
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
print("GPU:", gpu or "NONE - switch Runtime > Change runtime type to L4 or G4")
assert gpu, "No GPU attached"


In [ ]:
# 2. Google Drive and project folders
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/MSU-Alpha-3'
for sub in ['raw_video/test', 'registry', 'processed', 'labels', 'models', 'reports']:
    os.makedirs(f'{DRIVE}/{sub}', exist_ok=True)
print(sorted(os.listdir(DRIVE)))


In [ ]:
# 3. Repo and tests
REPO = '/content/MSU-alpha-tree'
if not os.path.exists(REPO):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/aldemirkonuk/MSU-alpha-tree.git', REPO], check=True)
else:
    subprocess.run(['git', '-C', REPO, 'pull', '-q'], check=True)
subprocess.run(['pip', 'install', '-q', 'pytest', 'pyyaml'], check=True)
t = subprocess.run(['python', '-m', 'pytest', '-q', REPO + '/tests'], capture_output=True, text=True)
print(t.stdout[-400:])
tests_passed = t.returncode == 0
assert tests_passed


In [ ]:
# 4. Test clip: first 120 s of the public PlayOn sample (pipeline testing only)
CLIP = f'{DRIVE}/raw_video/test/playon_sample_120s.mp4'
SRC = 'https://d1l0eyz2lfj4xa.cloudfront.net/playOnPoly/693d86be47fd95f56c93ef98/cloud_hls/0_hd_hls.m3u8'
if not os.path.exists(CLIP):
    subprocess.run(['ffmpeg', '-loglevel', 'error', '-y', '-i', SRC, '-t', '120', '-c', 'copy', '-bsf:a', 'aac_adtstoasc', CLIP], check=True)
import cv2
cap = cv2.VideoCapture(CLIP)
fps = cap.get(cv2.CAP_PROP_FPS); n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
w, h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
print(f'{w}x{h} @ {fps:.2f} fps, {n} frames, {os.path.getsize(CLIP)/1e6:.1f} MB')


In [ ]:
# 5. Pretrained RF-DETR person detection on 1 frame per second
subprocess.run(['pip', 'install', '-q', 'rfdetr', 'supervision'], check=True)
from rfdetr import RFDETRBase
from PIL import Image
import numpy as np
model = RFDETRBase()
try:
    from rfdetr.util.coco_classes import COCO_CLASSES
    PERSON = next(k for k, v in COCO_CLASSES.items() if v == 'person')
except Exception:
    PERSON = 1  # fallback: COCO 'person'
print('person class id:', PERSON)
step = max(int(round(fps)), 1)
counts, times = [], []
cap = cv2.VideoCapture(CLIP)
for i in range(0, n, step):
    cap.set(cv2.CAP_PROP_POS_FRAMES, i)
    ok, frame = cap.read()
    if not ok:
        break
    img = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    t0 = time.time()
    det = model.predict(img, threshold=0.4)
    times.append(time.time() - t0)
    counts.append(int((det.class_id == PERSON).sum()))
counts = np.array(counts)
print('frames scored:', len(counts))
print('people per frame: median', np.median(counts), 'min', counts.min(), 'max', counts.max())
print('share of frames with >= 8 people:', round(float((counts >= 8).mean()), 3))
print('seconds per frame (median):', round(float(np.median(times)), 4))


In [ ]:
# 6. Metrics to the repo and Drive
metrics = {
  'notebook': '00_smoke_test', 'ran_at_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
  'gpu': gpu, 'tests_passed': tests_passed,
  'clip': {'width': w, 'height': h, 'fps': round(fps, 3), 'frames': n, 'mb': round(os.path.getsize(CLIP)/1e6, 1)},
  'detector': 'RF-DETR Base (COCO pretrained), threshold 0.4, 1 frame/s',
  'people_per_frame': {'median': float(np.median(counts)), 'min': int(counts.min()), 'max': int(counts.max())},
  'share_frames_ge_8_people': round(float((counts >= 8).mean()), 3),
  'sec_per_frame_median': round(float(np.median(times)), 4),
  'est_gpu_hours_per_40min_game_at_full_fps': round(float(np.median(times)) * fps * 2400 / 3600, 2),
}
os.makedirs(f'{REPO}/results/00_smoke', exist_ok=True)
for path in [f'{REPO}/results/00_smoke/metrics.json', f'{DRIVE}/reports/00_smoke_metrics.json']:
    with open(path, 'w') as f: json.dump(metrics, f, indent=2)
print(json.dumps(metrics, indent=2))


**After the run (Mac session):** copy `results/00_smoke/metrics.json` into the local repo, commit with a message naming the GPU, and push. If any cell failed, commit `results/00_smoke/error.txt` with the cell number and the full error instead.